©2026. For information, contact Deloitte Tohmatsu Group.

# 📝 演習概要

この演習では、LSTM（Long Short-Term Memory）をNumPy/CuPyでスクラッチ実装します。忘却ゲート・入力ゲート・出力ゲートの各機構をコードで再現し、時系列データに対するモデル構築と学習を行うことで、LSTMが長期依存関係を学習できる仕組みを理解します。

# 事前準備

[JDLAが策定しているバージョン](https://www.jdla.org/certificate/engineer/)に合わせるために、以下のセルの実行をお願いします．

（#コメントアウト されているものは必要ありません）

また実行完了後に「ランタイムの再起動」をして下さい．

（以下のセルの実行は、最初にしていただければ、以降必要ありません．）

In [ ]:
# %%capture
# !pip uninstall matplotlib -y
# !pip install matplotlib==3.9.4

# !pip uninstall opencv-python -y
# !pip install opencv-python==4.11.0.86

# !pip uninstall torch -y
# !pip install torch==2.7.0

# !pip uninstall torchvision -y
# !pip install torchvision==0.22.0

## 時系列データに対応したLSTMレイヤをPythonで実装、モデルを構築し学習する

### GPUの設定

In [ ]:
#GPUの設定
GPU = True
#GPU = False

if GPU:
    import cupy as np
    import cupyx
    np.cuda.set_allocator(np.cuda.MemoryPool().malloc)
    # GPUモード: NumPy互換のCuPyを np として使う

    print('\033[92m' + '-' * 60 + '\033[0m')
    print(' ' * 23 + '\033[92mGPU Mode (cupy)\033[0m')
    print('\033[92m' + '-' * 60 + '\033[0m\n')
else:
    import numpy as np
    # CPUモード: NumPyを使用。

------------------------------------------------------------
                       GPU Mode (cupy)
------------------------------------------------------------



In [ ]:
# CuPy配列→NumPy配列へ変換
def to_cpu(x):
    import numpy
    if type(x) == numpy.ndarray:
        return x
    return np.asnumpy(x)

# NumPy配列→CuPy配列へ変換
def to_gpu(x):
    import cupy
    import cupyx
    if type(x) == cupy.ndarray:
        return x
    return cupy.asarray(x)

### 利用する基本的な層クラスの設定

In [ ]:
# Truncated BPTT対応のTimeLSTM（時系列方向にLSTMセルを展開）
class TimeLSTM:
    def __init__(self, Wx, Wh, b, stateful=False):
        # params: 重みリスト [Wx, Wh, b]
        self.params = [Wx, Wh, b]
        # grads: 勾配リスト（ゼロ初期化・同形状）
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        # 各タイムステップのLSTMセル（逆伝播で参照）
        self.layers = None

        # 隠れ状態hとセル状態c（stateful=Trueのときバッチ間で保持）
        self.h, self.c = None, None
        # 最終タイムステップのdh（上流からの勾配連結用）
        self.dh = None
        self.stateful = stateful

    def forward(self, xs):
        # xs: (N, T, D)  N:バッチ, T:系列長, D:入力次元
        Wx, Wh, b = self.params
        N, T, D = xs.shape
        H = Wh.shape[0]  # 隠れ状態次元

        self.layers = []
        # 出力隠れ状態系列 hs を格納 (N, T, H)
        hs = np.empty((N, T, H), dtype='f')

        # statefulがFalse、または初回はh,cをゼロで初期化
        if not self.stateful or self.h is None:
            self.h = np.zeros((N, H), dtype='f')
        if not self.stateful or self.c is None:
            self.c = np.zeros((N, H), dtype='f')

        # 時間方向にLSTMセルを展開
        for t in range(T):
            layer = LSTM(*self.params)  # 1ステップ分のLSTMセル
            self.h, self.c = layer.forward(xs[:, t, :], self.h, self.c)
            hs[:, t, :] = self.h
            self.layers.append(layer)

        return hs  # (N, T, H)

    def backward(self, dhs):
        # dhs: (N, T, H) 出力隠れ状態に対する損失の勾配
        Wx, Wh, b = self.params
        N, T, H = dhs.shape
        D = Wx.shape[0]

        # 入力側へ返す勾配の器 (N, T, D)
        dxs = np.empty((N, T, D), dtype='f')
        # 次タイムステップから伝わる勾配（初期は0）
        dh, dc = 0, 0

        # パラメータ勾配蓄積用
        grads = [0, 0, 0]

        # 時間を逆順に走査（BPTT）
        for t in reversed(range(T)):
            layer = self.layers[t]
            # dh: 現ステップ分の勾配と次ステップからの伝搬を合算
            dx, dh, dc = layer.backward(dhs[:, t, :] + dh, dc)
            dxs[:, t, :] = dx
            # 各セルの勾配を合算
            for i, grad in enumerate(layer.grads):
                grads[i] += grad

        # 勾配をメンバに反映
        for i, grad in enumerate(grads):
            self.grads[i][...] = grad
        self.dh = dh  # 先頭時刻へ伝わる勾配
        return dxs  # (N, T, D)

    def set_state(self, h, c=None):
        # 外部からh,cを与える（連続ストリーム入力時など）
        self.h, self.c = h, c

    def reset_state(self):
        # h,cを破棄（ミニバッチ間の依存を断つ）
        self.h, self.c = None, None


In [ ]:
# Embeddingの設定
# 単語ID→埋め込みベクトル
class Embedding:
    def __init__(self, W):
        # W: (V, D) V:語彙数, D:埋め込み次元
        self.params = [W]
        self.grads = [np.zeros_like(W)]
        self.idx = None  # 取り出したインデックスを保持

    def forward(self, idx):
        # idx: 整数ID配列（形状は自由）→ out: (..., D)
        W, = self.params
        self.idx = idx
        out = W[idx]
        return out

    def backward(self, dout):
        # dout: W[idx]に対応する勾配（outと同形）
        dW, = self.grads
        dW[...] = 0  # 勾配をいったんゼロクリア
        if GPU:
            # CuPy高速版のインデックス加算
            cupyx.scatter_add(dW, self.idx, dout)
        else:
            # NumPyの in-place 累積加算
            np.add.at(dW, self.idx, dout)
        return None

# 時系列版Embedding（各時刻にEmbeddingを適用）
class TimeEmbedding:
    def __init__(self, W):
        self.params = [W]
        self.grads = [np.zeros_like(W)]
        self.layers = None
        self.W = W

    def forward(self, xs):
        # xs: (N, T)  各時刻の単語ID
        N, T = xs.shape
        V, D = self.W.shape

        out = np.empty((N, T, D), dtype='f')
        self.layers = []

        # 各時刻にEmbeddingレイヤを適用
        for t in range(T):
            layer = Embedding(self.W)
            out[:, t, :] = layer.forward(xs[:, t])
            self.layers.append(layer)

        return out  # (N, T, D)

    def backward(self, dout):
        # dout: (N, T, D)
        N, T, D = dout.shape

        grad = 0
        # 各時刻の勾配を語彙埋め込みに累積
        for t in range(T):
            layer = self.layers[t]
            layer.backward(dout[:, t, :])
            grad += layer.grads[0]

        self.grads[0][...] = grad
        return None


In [ ]:
# TimeAffine層の設定
# 時系列全結合（各時刻の特徴に同じW,bを適用）
class TimeAffine:
    def __init__(self, W, b):
        # W: (D, M), b: (M,)
        self.params = [W, b]
        self.grads = [np.zeros_like(W), np.zeros_like(b)]
        self.x = None  # 逆伝播用に入力を保持

    def forward(self, x):
        # x: (N, T, D) → out: (N, T, M)
        N, T, D = x.shape
        W, b = self.params

        rx = x.reshape(N*T, -1)  # 時刻とバッチをまとめて行列計算
        out = np.dot(rx, W) + b
        self.x = x
        return out.reshape(N, T, -1)

    def backward(self, dout):
        # dout: (N, T, M) → dx: (N, T, D)
        x = self.x
        N, T, D = x.shape
        W, b = self.params

        dout = dout.reshape(N*T, -1)
        rx = x.reshape(N*T, -1)

        db = np.sum(dout, axis=0)
        dW = np.dot(rx.T, dout)
        dx = np.dot(dout, W.T)
        dx = dx.reshape(*x.shape)

        self.grads[0][...] = dW
        self.grads[1][...] = db

        return dx

In [ ]:
# TimeSoftmaxWithLoss層
# 時系列Softmax + 損失（ignore_labelをマスク）
class TimeSoftmaxWithLoss:
    def __init__(self):
        self.params, self.grads = [], []
        self.cache = None
        # このラベル値は損失/勾配計算から除外（パディング用など）
        self.ignore_label = -1

    def forward(self, xs, ts):
        # xs: (N, T, V) ロジット, ts: (N, T) もしくは one-hot (N, T, V)
        N, T, V = xs.shape

        if ts.ndim == 3:
            # one-hotの場合はargmaxでクラスIDに変換
            ts = ts.argmax(axis=2)

        # マスク（ignore_labelを除外）
        mask = (ts != self.ignore_label)

        # (N*T, V) に平坦化して計算を簡略化
        xs = xs.reshape(N * T, V)
        ts = ts.reshape(N * T)
        mask = mask.reshape(N * T)

        # 安定化Softmax→対数尤度の合計（マスク適用）
        ys = softmax(xs)
        ls = np.log(ys[np.arange(N * T), ts])
        ls *= mask
        loss = -np.sum(ls)
        # 有効サンプル数で平均化
        loss /= mask.sum()

        # 逆伝播用に保持
        self.cache = (ts, ys, mask, (N, T, V))
        return loss

    def backward(self, dout=1):
        # doutは通常1（dL/dL）
        ts, ys, mask, (N, T, V) = self.cache

        dx = ys
        dx[np.arange(N * T), ts] -= 1  # Softmax+CEの微分
        dx *= dout
        dx /= mask.sum()               # 平均化
        dx *= mask[:, np.newaxis]      # ignore部分をゼロ化

        dx = dx.reshape((N, T, V))
        return dx

### 活性関数の設定

In [ ]:
#活性関数
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def relu(x):
    return np.maximum(0, x)


def softmax(x):
    # 数値安定化のために最大値を引く（2次元・1次元の両対応）
    if x.ndim == 2:
        x = x - x.max(axis=-1, keepdims=True)
        x = np.exp(x)
        x /= x.sum(axis=-1, keepdims=True)
    elif x.ndim == 1:
        x = x - np.max(x)
        x = np.exp(x) / np.sum(np.exp(x))

    return x

### 最適化手法の設定

In [ ]:
#  最適化手法の設定
#  確率的勾配降下法（Stochastic Gradient Descent）
class SGD:
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, params, grads):
        # params, gradsは同じ順序のリスト
        for i in range(len(params)):
            params[i] -= self.lr * grads[i]

## データセットの取得

In [ ]:
# データセットの獲得
import os
try:
    import urllib.request
except ImportError:
    raise ImportError('Use Python3!')
import pickle
# import numpy as np

# データURLとファイル名マップ
url_base = 'https://raw.githubusercontent.com/tomsercu/lstm/master/data/'
key_file = {
    'train':'ptb.train.txt',
    'test':'ptb.test.txt',
    'valid':'ptb.valid.txt'
}
save_file = {
    'train':'ptb.train.npy',
    'test':'ptb.test.npy',
    'valid':'ptb.valid.npy'
}
vocab_file = 'ptb.vocab.pkl'

# 保存先ディレクトリ
dataset_dir = os.getcwd()
# os.path.dirname(os.path.abspath(__file__))


def _download(file_name):
    # 存在チェックして未ダウンロードなら取得
    file_path = dataset_dir + '/' + file_name
    if os.path.exists(file_path):
        return

    print('Downloading ' + file_name + ' ... ')

    try:
        urllib.request.urlretrieve(url_base + file_name, file_path)
    except urllib.error.URLError:
        import ssl
        ssl._create_default_https_context = ssl._create_unverified_context
        urllib.request.urlretrieve(url_base + file_name, file_path)

    print('Done')


def load_vocab():
    # 語彙辞書の保存パス
    vocab_path = dataset_dir + '/' + vocab_file

    if os.path.exists(vocab_path):
        # 既存の語彙をロード
        with open(vocab_path, 'rb') as f:
            word_to_id, id_to_word = pickle.load(f)
        return word_to_id, id_to_word


    # 語彙が無い場合はtrainから語彙を構築
    word_to_id = {}
    id_to_word = {}
    data_type = 'train'
    file_name = key_file[data_type]
    file_path = dataset_dir + '/' + file_name

    _download(file_name)

    # 改行を<eos>に置換しスペース区切りで分割
    words = open(file_path).read().replace('\n', '<eos>').strip().split()

    # 単語→ID、ID→単語を作成
    for i, word in enumerate(words):
        if word not in word_to_id:
            tmp_id = len(word_to_id)
            word_to_id[word] = tmp_id
            id_to_word[tmp_id] = word


    # 語彙をピクルで保存
    with open(vocab_path, 'wb') as f:
        pickle.dump((word_to_id, id_to_word), f)

    return word_to_id, id_to_word


def load_data(data_type='train'):
    '''
        :param data_type: データの種類：'train' or 'test' or 'valid (val)'
        :return:
    '''
    if data_type == 'val': data_type = 'valid'
    save_path = dataset_dir + '/' + save_file[data_type]

    word_to_id, id_to_word = load_vocab()

    if os.path.exists(save_path):
        # 前回保存済みのID列をロード
        corpus = np.load(save_path)
        return corpus, word_to_id, id_to_word


    # 未保存ならテキストから作成
    file_name = key_file[data_type]
    file_path = dataset_dir + '/' + file_name
    _download(file_name)

    words = open(file_path).read().replace('\n', '<eos>').strip().split()
    # 単語列→ID列に変換（np.arrayにキャスト）
    corpus = np.array([word_to_id[w] for w in words])

    # 次回以降の高速化のため保存
    np.save(save_path, corpus)
    return corpus, word_to_id, id_to_word


# 直接実行時の動作（train/val/testをすべて準備）
if __name__ == '__main__':
    for data_type in ('train', 'val', 'test'):
        load_data(data_type)

Done
Done
Done


### 勾配爆発への対策

In [ ]:
#clip_gradsの設定
#勾配クリッピング(勾配爆発への対策)
#勾配が閾値を超えた場合に、勾配を修正するもの。

import matplotlib.pyplot as plt
dW1 = np.random.rand(3, 3) * 10
dW2 = np.random.rand(3, 3) * 10
grads = [dW1, dW2]
max_norm = 5.0

def clip_grads(grads, max_norm):
    # 全勾配のL2ノルムを計算して上限にスケール
    total_norm = 0
    for grad in grads:
        total_norm += np.sum(grad ** 2)
    total_norm = np.sqrt(total_norm)

    rate = max_norm / (total_norm + 1e-6)
    if rate < 1:
        for grad in grads:
            grad *= rate  # クリッピング（in-place）

clip_grads(grads, max_norm)

In [ ]:
def remove_duplicate(params, grads):
    '''
    パラメータ配列中の重複する重みをひとつに集約し、
    その重みに対応する勾配を加算する
    '''
    params, grads = params[:], grads[:]  # copy list

    while True:
        find_flg = False
        L = len(params)

        for i in range(0, L - 1):
            for j in range(i + 1, L):
                # 重みを共有する場合
                if params[i] is params[j]:
                    grads[i] += grads[j]  # 勾配の加算
                    find_flg = True
                    params.pop(j)
                    grads.pop(j)
                # 転置行列として重みを共有する場合（weight tying）
                elif params[i].ndim == 2 and params[j].ndim == 2 and \
                     params[i].T.shape == params[j].shape and np.all(params[i].T == params[j]):
                    grads[i] += grads[j].T
                    find_flg = True
                    params.pop(j)
                    grads.pop(j)

                if find_flg: break
            if find_flg: break

        if not find_flg: break

    return params, grads

### ベースモデルの作成

In [ ]:
# BaseModelの設定
# モデル基底クラス
class BaseModel:
    def __init__(self):
        self.params, self.grads = None, None

    def forward(self, *args):
        raise NotImplementedError

    def backward(self, *args):
        raise NotImplementedError

    def save_params(self, file_name=None):
        # モデルパラメータをfp16で軽量保存（GPU時はCPUに戻して保存）
        if file_name is None:
            file_name = self.__class__.__name__ + '.pkl'

        params = [p.astype(np.float16) for p in self.params]
        if GPU:
            params = [to_cpu(p) for p in params]

        with open(file_name, 'wb') as f:
            pickle.dump(params, f)

    def load_params(self, file_name=None):
        # 保存したパラメータをロード（GPU時はGPU配列に戻す）
        if file_name is None:
            file_name = self.__class__.__name__ + '.pkl'

        if '/' in file_name:
            file_name = file_name.replace('/', os.sep)

        if not os.path.exists(file_name):
            raise IOError('No file: ' + file_name)

        with open(file_name, 'rb') as f:
            params = pickle.load(f)

        params = [p.astype('f') for p in params]
        if GPU:
            params = [to_gpu(p) for p in params]

        for i, param in enumerate(self.params):
            param[...] = params[i]


## LSTM

LSTMモデルの設定/勾配消失に対応するためのモデル/ゲート付きRNNの代表格

In [ ]:
class LSTM:
    def __init__(self, Wx, Wh, b):
        '''
        Parameters
        ----------
        Wx: 入力`x`用の重みパラーメタ（4つ分の重みをまとめる）
        Wh: 隠れ状態`h`用の重みパラメータ（4つ分の重みをまとめる）
        b: バイアス（4つ分のバイアスをまとめる）
        '''
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.cache = None  # 逆伝播用の中間結果

    def forward(self, x, h_prev, c_prev):
        # x: (N, D), h_prev: (N, H), c_prev: (N, H)
        Wx, Wh, b = self.params
        N, H = h_prev.shape

        A = np.dot(x, Wx) + np.dot(h_prev, Wh) + b
        #４つ分のアフィン変換の結果が格納されているため、sliceする必要がある。
        f = A[:, :H]         # forget
        g = A[:, H:2*H]      # candidate
        i = A[:, 2*H:3*H]    # input
        o = A[:, 3*H:]       # output

        # 非線形変換
        f = sigmoid(f)
        g = np.tanh(g)
        i = sigmoid(i)
        o = sigmoid(o)

        # セル状態と隠れ状態の更新
        c_next = f * c_prev + g * i
        h_next = o * np.tanh(c_next)

        # 逆伝播に必要な値をキャッシュ
        self.cache = (x, h_prev, c_prev, i, f, g, o, c_next)

        return h_next, c_next  # (N, H), (N, H)

    def backward(self, dh_next, dc_next):
        # dh_next, dc_next: 次時刻からの勾配
        Wx, Wh, b = self.params
        x, h_prev, c_prev, i, f, g, o, c_next = self.cache

        tanh_c_next = np.tanh(c_next)

        # セル経由の勾配合成
        ds = dc_next + (dh_next * o) * (1 - tanh_c_next ** 2)

        # 各項目への勾配
        dc_prev = ds * f

        di = ds * g
        df = ds * c_prev
        do = dh_next * tanh_c_next
        dg = ds * i

        # 活性化の微分を掛ける
        di *= i * (1 - i)
        df *= f * (1 - f)
        do *= o * (1 - o)
        dg *= (1 - g ** 2)

        # 4ゲートを結合して元の次元へ
        dA = np.hstack((df, dg, di, do))

        # パラメータ勾配
        dWh = np.dot(h_prev.T, dA)
        dWx = np.dot(x.T, dA)
        db = dA.sum(axis=0)

        self.grads[0][...] = dWx
        self.grads[1][...] = dWh
        self.grads[2][...] = db

        # 入力側と前時刻隠れ状態への勾配
        dx = np.dot(dA, Wx.T)
        dh_prev = np.dot(dA, Wh.T)

        return dx, dh_prev, dc_prev


### LSTMの学習モデルの構築

In [ ]:
#モデルLSTMlmの設定

class LSTMlm:
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        #重みの初期化
        embed_W = (rn(V, D) / 100).astype('f')
        lstm_Wx = (rn(D, 4 * H) / np.sqrt(D).astype('f'))
        lstm_Wh = (rn(H, 4 * H) / np.sqrt(H).astype('f'))
        lstm_b = np.zeros(4 * H).astype('f')
        affine_W = (rn(H, V) / np.sqrt(H)).astype('f')
        affine_b = np.zeros(V).astype('f')

        #レイヤの生成
        self.layers = [
            TimeEmbedding(embed_W),
            TimeLSTM(lstm_Wx, lstm_Wh, lstm_b, stateful=True),
            TimeAffine(affine_W, affine_b)
        ]
        self.loss_layer = TimeSoftmaxWithLoss()
        self.lstm_layer = self.layers[1]  # 状態リセット用に参照

        #すべての重みと勾配をリストにまとめる
        self.params, self.grads = [], []
        for layer in self.layers:
            self.params += layer.params
            self.grads += layer.grads

    def forward(self, xs, ts):
        # xs: (N, T) 入力ID列, ts: (N, T) 正解ID列
        for layer in self.layers:
            xs = layer.forward(xs)
        loss = self.loss_layer.forward(xs, ts)
        return loss

    def backward(self, dout=1):
        # 逆順に勾配を伝搬
        dout = self.loss_layer.backward(dout)
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def reset_state(self):
        # 学習/評価の区切りでLSTM状態をリセット
        self.lstm_layer.reset_state()


### モデルの学習

In [ ]:
import matplotlib.pyplot as plt
# import numpy as np

#ハイパーパラメータの設定
batch_size = 10
wordvec_size = 100
hidden_size = 100 #RNNの隠れ状態ベクトルの要素数
time_size = 5 #Truncated BPTTの展開する時間サイズ
lr = 0.1
max_epoch = 50
max_grad = 0.25    # 勾配クリッピングの上限

#学習データの読み込み（データを小さくする）
corpus, word_to_id, id_to_word = load_data('train')
corpus_size = 1000
corpus = corpus[:corpus_size]
vocab_size = int(max(corpus) + 1)

xs = corpus[:-1]#入力
ts = corpus[1:]#出力（教師ラベル）
data_size = len(xs)
print('corpus size: %d, vocabulary size: %d' % (corpus_size, vocab_size))

#学習時に使用する変数
max_iters = data_size // (batch_size * time_size)
time_idx = 0
total_loss = 0
loss_count = 0
ppl_list = []  # Perplexity推移の記録

# モデルと最適化手法の用意
model = LSTMlm(vocab_size, wordvec_size, hidden_size)
optimizer = SGD(lr)

#①ミニパッチの各サンプルの読み込み開始位置を計算
jump = (corpus_size - 1) // batch_size
offsets = [i * jump for i in range(batch_size)]

for epoch in range(max_epoch):
    for iter in range(max_iters):
        #②ミニバッチの取得
        batch_x = np.empty((batch_size, time_size), dtype='i')
        batch_t = np.empty((batch_size, time_size), dtype='i')

        # 時間方向にTBPTT長分だけ切り出す
        for t in range(time_size):
            for i, offset in enumerate(offsets):
                batch_x[i, t] = xs[(offset + time_idx) % data_size]
                batch_t[i, t] = ts[(offset + time_idx) % data_size]
                time_idx += 1  # 逐次的に読み取り位置を進める
        # 順伝播→損失
        loss = model.forward(batch_x, batch_t)
        # 逆伝播
        model.backward()
        # パラメータ更新（SGD）
        optimizer.update(model.params, model.grads)
        # ロスの集計
        total_loss += loss
        loss_count += 1

    #③エポックごとにパープレキシティの評価
    ppl = np.exp(total_loss / loss_count)
    print('| epoch %d | perplexity %.2f' % (epoch+1, ppl))
    ppl_list.append(float(ppl))
    total_loss, loss_count = 0, 0


corpus size: 1000, vocabulary size: 418
| epoch 1 | perplexity 406.66
| epoch 2 | perplexity 389.33
| epoch 3 | perplexity 373.87
| epoch 4 | perplexity 378.01
| epoch 5 | perplexity 366.52
| epoch 6 | perplexity 343.58
| epoch 7 | perplexity 322.63
| epoch 8 | perplexity 331.66
| epoch 9 | perplexity 314.85
| epoch 10 | perplexity 293.78
| epoch 11 | perplexity 260.00
| epoch 12 | perplexity 215.66
| epoch 13 | perplexity 185.25
| epoch 14 | perplexity 229.56
| epoch 15 | perplexity 247.57
| epoch 16 | perplexity 211.97
| epoch 17 | perplexity 191.88
| epoch 18 | perplexity 195.97
| epoch 19 | perplexity 214.27
| epoch 20 | perplexity 234.19
| epoch 21 | perplexity 213.96
| epoch 22 | perplexity 159.88
| epoch 23 | perplexity 162.42
| epoch 24 | perplexity 169.22
| epoch 25 | perplexity 232.34
| epoch 26 | perplexity 216.14
| epoch 27 | perplexity 174.94
| epoch 28 | perplexity 168.89
| epoch 29 | perplexity 203.27
| epoch 30 | perplexity 214.65
| epoch 31 | perplexity 209.57
| epoch 

## 🔧 実践問題1：忘却ゲートのバイアスを1.0に初期化して学習を改善する

上のコードでは `lstm_b = np.zeros(4 * H)` でバイアスを全て0に初期化しています。
しかし、LSTMでは**忘却ゲート（forget gate）のバイアスを1.0で初期化する**のが広く知られた改善手法です。

理由：
- 忘却ゲートの出力は `sigmoid(f)` で計算される
- バイアス=0 だと `sigmoid(0) = 0.5` → 学習初期から過去の情報を半分忘れてしまう
- バイアス=1 だと `sigmoid(1) ≈ 0.73` → 学習初期は過去の情報を**より多く保持**する

LSTMのバイアスベクトル `b` は `4*H` 次元で、以下の順にゲートが並んでいます：

```
b = [ forget(H個) | candidate(H個) | input(H個) | output(H個) ]
     ^^^^^^^^^^^^^^^^
     ここだけ1.0にしたい
```

---

**問題：** 以下のコードの `______` を埋めて、忘却ゲート部分のバイアスだけを1.0に初期化し、パープレキシティの改善を確認してください。

<br/>

<details>
<summary>💡 <b>ヒント（クリックして表示）</b></summary>

<blockquote>
忘却ゲートは <code>b</code> の最初のH個の要素です。NumPyのスライス <code>b[:H]</code> で指定できます。
</blockquote>

</details>

<br/>


In [ ]:
# 忘却ゲートのバイアスを1.0に初期化したLSTMlmを作成

H = 100  # hidden_size

model2 = LSTMlm(vocab_size, wordvec_size, H)

# LSTMの重みはmodel2.params内にある
# params = [embed_W, lstm_Wx, lstm_Wh, lstm_b, affine_W, affine_b]
# lstm_b は params[3] にあたる

lstm_b = model2.params[3]  # shape: (4*H,)
print(f'lstm_b shape: {lstm_b.shape}')  # (400,) を確認

# 忘却ゲートのバイアス（最初のH個）だけを1.0にする
lstm_b[:______] = ______

# 残りのゲート（candidate, input, output）はそのまま0
print(f'forget gate bias mean: {lstm_b[:H].mean():.1f}')  # 1.0 と表示されるはず
print(f'other gates bias mean: {lstm_b[H:].mean():.1f}')  # 0.0 と表示されるはず

# 学習（元と同じ設定）
optimizer2 = SGD(lr)
time_idx2 = 0; total_loss2 = 0; loss_count2 = 0; ppl_list2 = []
jump2 = (corpus_size - 1) // batch_size
offsets2 = [i * jump2 for i in range(batch_size)]

for epoch in range(max_epoch):
    for iter in range(max_iters):
        batch_x = np.empty((batch_size, time_size), dtype='i')
        batch_t = np.empty((batch_size, time_size), dtype='i')
        for t in range(time_size):
            for i, offset in enumerate(offsets2):
                batch_x[i, t] = xs[(offset + time_idx2) % data_size]
                batch_t[i, t] = ts[(offset + time_idx2) % data_size]
            time_idx2 += 1
        loss = model2.forward(batch_x, batch_t)
        model2.backward()
        params2 = [p for p in model2.params]
        grads2 = [g for g in model2.grads]
        clip_grads(grads2, max_grad)
        optimizer2.update(params2, grads2)
        total_loss2 += loss; loss_count2 += 1
    ppl = np.exp(total_loss2 / loss_count2)
    if (epoch+1) % 10 == 0:
        print(f'| epoch {epoch+1} | perplexity {ppl:.2f}')
    ppl_list2.append(float(ppl))
    total_loss2, loss_count2 = 0, 0

plt.plot(ppl_list, label='bias=0 (default)', alpha=0.7)
plt.plot(ppl_list2, label='forget bias=1.0', alpha=0.7)
plt.xlabel('Epochs'); plt.ylabel('Perplexity')
plt.legend(); plt.title('Forget gate bias initialization'); plt.show()


<details><summary>解答例</summary>

```python
lstm_b[:H] = 1.0
```

- `lstm_b` は `(4*H,)` = `(400,)` のベクトルで、最初のH個（`[:100]`）が忘却ゲートのバイアスです
- `sigmoid(1.0) ≈ 0.73` なので、学習初期に忘却ゲートが「ほぼ開いた」状態になり、セル状態（過去の情報）がより多く保持されます
- これにより長距離の依存関係を学習しやすくなり、特に学習初期のパープレキシティの下がり方が速くなる傾向があります
- この手法は論文 [Jozefowicz et al., 2015](http://proceedings.mlr.press/v37/jozefowicz15.pdf) で推奨されており、PyTorchの `nn.LSTM` でもデフォルトで忘却ゲートバイアスを考慮した初期化が行われています
</details>


## 🔧 実践問題2：LSTMを2層に積んで（Stacked LSTM）精度を比較する

上のLSTMlmモデルは1層のLSTMで構成されています。
LSTMを複数層に積む（stacking）ことで、より抽象的な特徴を段階的に学習できます。

2層構成にするには、もう1つ `TimeLSTM` を追加します。このとき重みの次元に注意してください：

```
1層目: 入力(D次元) → 隠れ(H次元)
   Wx1: (D, 4*H)   Wh1: (H, 4*H)   b1: (4*H,)

2層目: 入力(???次元) → 隠れ(H次元)
   Wx2: (?, 4*H)   Wh2: (H, 4*H)   b2: (4*H,)
```

---

**問題：** 以下のコードの `______` を埋めて、2層LSTMの言語モデルを作成してください。

<br/>

<details>
<summary>💡 <b>ヒント（クリックして表示）</b></summary>

<blockquote>
2層目のLSTMの「入力」は、1層目のLSTMの「出力（隠れ状態）」です。1層目の出力次元は何次元でしょうか？
</blockquote>

</details>

<br/>


In [ ]:
class StackedLSTMlm:
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        # Embedding
        embed_W = (rn(V, D) / 100).astype('f')

        # 1層目 LSTM: 入力D次元 → 隠れH次元
        lstm1_Wx = (rn(D, 4 * H) / np.sqrt(D)).astype('f')
        lstm1_Wh = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm1_b = np.zeros(4 * H).astype('f')

        # 2層目 LSTM: 入力 ______ 次元 → 隠れH次元
        # 1層目の出力（隠れ状態）が2層目の入力になる
        lstm2_Wx = (rn(______, 4 * H) / np.sqrt(______)).astype('f')
        lstm2_Wh = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm2_b = np.zeros(______ * H).astype('f')

        # Affine: 隠れH次元 → 語彙V次元
        affine_W = (rn(H, V) / np.sqrt(H)).astype('f')
        affine_b = np.zeros(V).astype('f')

        # レイヤの生成（4層構成）
        self.layers = [
            TimeEmbedding(embed_W),
            TimeLSTM(lstm1_Wx, lstm1_Wh, lstm1_b, stateful=True),
            TimeLSTM(lstm2_Wx, lstm2_Wh, lstm2_b, stateful=______),
            TimeAffine(affine_W, affine_b)
        ]
        self.loss_layer = TimeSoftmaxWithLoss()

        self.params, self.grads = [], []
        for layer in self.layers:
            self.params += layer.params
            self.grads += layer.grads

    def forward(self, xs, ts):
        for layer in self.layers:
            xs = layer.forward(xs)
        return self.loss_layer.forward(xs, ts)

    def backward(self, dout=1):
        dout = self.loss_layer.backward(dout)
        for layer in ______(self.layers):
            dout = layer.backward(dout)
        return dout


# 学習
model3 = StackedLSTMlm(vocab_size, wordvec_size, hidden_size)
optimizer3 = SGD(lr)
time_idx3 = 0; total_loss3 = 0; loss_count3 = 0; ppl_list3 = []
jump3 = (corpus_size - 1) // batch_size
offsets3 = [i * jump3 for i in range(batch_size)]

for epoch in range(max_epoch):
    for iter in range(max_iters):
        batch_x = np.empty((batch_size, time_size), dtype='i')
        batch_t = np.empty((batch_size, time_size), dtype='i')
        for t in range(time_size):
            for i, offset in enumerate(offsets3):
                batch_x[i, t] = xs[(offset + time_idx3) % data_size]
                batch_t[i, t] = ts[(offset + time_idx3) % data_size]
            time_idx3 += 1
        loss = model3.forward(batch_x, batch_t)
        model3.backward()
        clip_grads(model3.grads, max_grad)
        optimizer3.update(model3.params, model3.grads)
        total_loss3 += loss; loss_count3 += 1
    ppl = np.exp(total_loss3 / loss_count3)
    if (epoch+1) % 10 == 0:
        print(f'| epoch {epoch+1} | perplexity {ppl:.2f}')
    ppl_list3.append(float(ppl))
    total_loss3, loss_count3 = 0, 0

plt.plot(ppl_list, label='1-layer LSTM', alpha=0.7)
plt.plot(ppl_list3, label='2-layer Stacked LSTM', alpha=0.7)
plt.xlabel('Epochs'); plt.ylabel('Perplexity')
plt.legend(); plt.title('Stacked LSTM comparison'); plt.show()


<details><summary>解答例</summary>

```python
# 2層目のLSTM重み
lstm2_Wx = (rn(H, 4 * H) / np.sqrt(H)).astype('f')  # 入力はH次元（1層目の出力）
lstm2_b = np.zeros(4 * H).astype('f')  # ゲート4つ分

# 2層目もstateful=True
TimeLSTM(lstm2_Wx, lstm2_Wh, lstm2_b, stateful=True)

# 逆伝播は逆順
for layer in reversed(self.layers):
```

- 2層目のLSTMの入力次元は **H**（1層目の隠れ状態の次元数）です。Embedding(D次元)とは異なるため、`Wx2` の形状は `(H, 4*H)` になります
- バイアスは `4 * H` 次元（4ゲート × H個）で、1層目と同じ構造です
- 2層目も `stateful=True` にすることで、バッチ間で隠れ状態を引き継ぎます
- 逆伝播は `reversed()` で層を逆順に辿ります。これは1層のときと同じ原理ですが、層が増えると勾配がより多くの層を通過するため、勾配クリッピングがより重要になります
- 小さなコーパス（1000語）では2層にしてもあまり差が出ないことがありますが、大規模データでは層を増やす効果が顕著に現れます
</details>
